# `explain_day`: how one day's VaR is built

`explain_day(date, run, reading)` reads the saved files of a run (nothing is re-trained) and lays out, in order, how the VaR of ONE forecast day came about: the model's per-ticker variances and the EWMA correlation that give the portfolio variance, sigma, the quantile of the residual window, VaR = -(mu_hat + sigma * q), and the realized loss.

In [ ]:
import pandas as pd
import run_store as rs
import walk_forward as wf

pd.set_option('display.max_colwidth', 80)
READING = wf.Reading(var_confidence=0.95, window_size=252, min_periods=30, test_confidence=0.95)

runs_3y = rs.load_runs(train_bars=756)
by_name = {r.name: r for r in runs_3y}
list(by_name)

## A long run (xgb_long): per-ticker variances + EWMA correlation

In [2]:
rs.explain_day('2024-03-15', by_name['xgb_long'], READING)

,value,note
item,,
window_id,11,the window whose model forecast this day
weight[AAPL],0.25,
model_variance[AAPL],0.000171,the diagonal of the portfolio covariance
ewma_variance[AAPL],0.000133,"the EWMA's own diagonal (not used by a model, only its correlation)"
weight[JPM],0.25,
model_variance[JPM],0.000133,the diagonal of the portfolio covariance
ewma_variance[JPM],0.000074,"the EWMA's own diagonal (not used by a model, only its correlation)"
weight[XOM],0.25,
model_variance[XOM],0.000127,the diagonal of the portfolio covariance


## The same day for the GARCH wide baseline (the model gives the portfolio variance directly: no ticker rows)

In [3]:
rs.explain_day('2024-03-15', by_name['garch_wide'], READING)

,value,note
item,,
window_id,11,the window whose model forecast this day
predicted_variance,0.000037,the model's portfolio variance for this day
predicted_sigma,0.006051,sqrt of it
mu_hat,0.0,
quantile_window_n,252,standardized residuals strictly before this day
quantile_window_first,2023-03-15,
quantile_window_last,2024-03-14,
quantile_q,-1.249512,0.05-quantile of that window
varForecast,0.007561,-(mu_hat + sigma * q)


## An exceedance day: the largest excess of xgb_long, side by side with the baseline

In [4]:
xgb_long, garch_wide = by_name['xgb_long'], by_name['garch_wide']
exceeding = xgb_long.exceeding_days(READING)
day = exceeding['excess'].idxmax()
print('largest exceedance of xgb_long:', day.date(), '| excess', round(exceeding.loc[day, 'excess'], 4))
side_by_side = pd.concat(
    {'xgb_long': rs.explain_day(day, xgb_long, READING)['value'], 'garch_wide': rs.explain_day(day, garch_wide, READING)['value']},
    axis=1,
)
side_by_side.loc[['window_id', 'predicted_variance', 'predicted_sigma', 'quantile_q', 'varForecast', 'loss', 'exceeded', 'excess']]

largest exceedance of xgb_long: 2020-03-16 | excess 0.0934


,xgb_long,garch_wide
item,,
window_id,7,7
predicted_variance,0.000037,0.003071
predicted_sigma,0.006115,0.055415
quantile_q,-2.700812,-2.170674
varForecast,0.016516,0.120287
loss,0.109937,0.109937
exceeded,True,False
excess,0.093421,-0.01035


The long run's per-ticker rows for that day (what the model said for each ticker, and the correlation it was combined with):

In [5]:
rs.explain_day(day, xgb_long, READING).iloc[:19]

,value,note
item,,
window_id,7,the window whose model forecast this day
weight[AAPL],0.25,
model_variance[AAPL],0.000037,the diagonal of the portfolio covariance
ewma_variance[AAPL],0.002827,"the EWMA's own diagonal (not used by a model, only its correlation)"
weight[JPM],0.25,
model_variance[JPM],0.000028,the diagonal of the portfolio covariance
ewma_variance[JPM],0.004203,"the EWMA's own diagonal (not used by a model, only its correlation)"
weight[XOM],0.25,
model_variance[XOM],0.000054,the diagonal of the portfolio covariance
